# Dataset v2: построение, аудит и скрининг эксперимента B

Notebook выполняет четыре шага в одном воспроизводимом запуске:

1. создаёт новые манифесты поверх BSDS500 из dataset v1;
2. проверяет заполненность, яркость, источники и отсутствие вырожденных изображений;
3. упаковывает `/kaggle/working/ai_phonetics_data_v2.zip`;
4. сравнивает `BaselineUNet` и `SharedDepthUNet` на одинаковом небольшом subset.

Dataset v1 не изменяется. Если аудит v2 не проходит, обучение намеренно не запускается.

In [ ]:
from __future__ import annotations

import csv
import gc
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFont, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset

try:
    from pytorch_msssim import ssim
except ImportError:
    print('Installing pytorch-msssim...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])
    from pytorch_msssim import ssim

INPUT_V1 = Path('/kaggle/input/ai-phonetics-data-v1')
OUTPUT_ROOT = Path('/kaggle/working/ai_phonetics_data_v2')
ARCHIVE_BASE = Path('/kaggle/working/ai_phonetics_data_v2')
AUDIT_DIR = Path('/kaggle/working/dataset_v2_audit')
SCREEN_DIR = Path('/kaggle/working/dataset_v2_screen')

IMAGE_SIZE = 128
SPLIT_COUNTS = {'train': 24_000, 'val': 2_000, 'test_internal': 2_000}
BSDS_SPLITS = {'train': 'train', 'val': 'val', 'test_internal': 'test'}
SPLIT_SEEDS = {'train': 4202, 'val': 314152, 'test_internal': 271822}
NATURAL_FRACTION = 0.50

OCCUPANCY_RANGES = {
    'tiny': (0.005, 0.030),
    'sparse': (0.030, 0.080),
    'medium': (0.080, 0.200),
    'dense': (0.200, 0.500),
}
OCCUPANCY_NAMES = tuple(OCCUPANCY_RANGES)
OCCUPANCY_WEIGHTS = np.array([0.10, 0.25, 0.35, 0.30], dtype=np.float64)
BACKGROUND_NAMES = ('dark', 'lifted', 'light')
BACKGROUND_WEIGHTS = np.array([0.55, 0.25, 0.20], dtype=np.float64)

SCREEN_TRAIN = 512
SCREEN_VAL = 128
SCREEN_EPOCHS = 6
BATCH_SIZE = 16
LR = 3e-4
WEIGHT_DECAY = 1e-4
DATA_SEED = 42
TRAIN_SEED = 42
NUM_WORKERS = min(2, os.cpu_count() or 1)
LOG_EVERY = 10

WAVELENGTH = 532e-9
PIXEL_PITCH = 10e-6
ZS = (24.1e-3, 25.3e-3, 26.5e-3, 27.7e-3)

def find_root(path: Path) -> Path:
    if (path / 'dataset_metadata.json').is_file():
        return path
    matches = list(path.rglob('dataset_metadata.json'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Cannot find exactly one dataset_metadata.json below {path}')
    return matches[0].parent

V1_ROOT = find_root(INPUT_V1)
V1_BSDS = V1_ROOT / 'BSDS500' / 'data'
if not all((V1_BSDS / 'images' / split).is_dir() for split in ('train', 'val', 'test')):
    raise FileNotFoundError('Dataset v1 does not contain BSDS500/data/images/{train,val,test}')

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
SCREEN_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(V1_BSDS, OUTPUT_ROOT / 'BSDS500' / 'data')
BSDS_ROOT = OUTPUT_ROOT / 'BSDS500' / 'data'
print('V1 root:', V1_ROOT)
print('V2 output:', OUTPUT_ROOT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'disabled')

## 1. Loader и генератор v2

Процедурные изображения получают целевой диапазон заполненности и режим фона. Натуральные изображения
используют многомасштабные квадратные кропы либо всю сцену с сохранением пропорций.
Все изображения по-прежнему независимо нормируются на собственный максимум.

In [ ]:
RESAMPLE = Image.Resampling.LANCZOS

def normalize_image(array: np.ndarray) -> torch.Tensor:
    values = torch.from_numpy(np.asarray(array, dtype=np.float32) / 255.0)
    maximum = values.amax()
    if maximum <= 1e-8:
        raise ValueError('All-black image cannot be normalized')
    return values / maximum

def font_for_size(size: int):
    try:
        return ImageFont.truetype('DejaVuSans.ttf', size=size)
    except OSError:
        return ImageFont.load_default()

def draw_primitive(mask: Image.Image, rng: np.random.Generator, remaining_fraction: float) -> None:
    draw = ImageDraw.Draw(mask)
    target_area = max(12.0, remaining_fraction * IMAGE_SIZE * IMAGE_SIZE * float(rng.uniform(0.25, 0.65)))
    kind = str(rng.choice(['ellipse', 'rectangle', 'line', 'ring', 'glyph'], p=[0.28, 0.28, 0.16, 0.14, 0.14]))
    aspect = float(np.exp(rng.uniform(np.log(0.45), np.log(2.2))))
    width = int(np.clip(np.sqrt(target_area * aspect), 3, IMAGE_SIZE - 4))
    height = int(np.clip(np.sqrt(target_area / aspect), 3, IMAGE_SIZE - 4))
    x0 = int(rng.integers(2, max(3, IMAGE_SIZE - width - 1)))
    y0 = int(rng.integers(2, max(3, IMAGE_SIZE - height - 1)))
    x1, y1 = min(IMAGE_SIZE - 2, x0 + width), min(IMAGE_SIZE - 2, y0 + height)

    if kind == 'ellipse':
        draw.ellipse((x0, y0, x1, y1), fill=255)
    elif kind == 'rectangle':
        draw.rectangle((x0, y0, x1, y1), fill=255)
    elif kind == 'line':
        line_width = max(1, int(np.clip(target_area / max(width, height), 1, 10)))
        draw.line((x0, y0, x1, y1), fill=255, width=line_width)
    elif kind == 'ring':
        ring_width = max(1, int(np.clip(min(width, height) * 0.12, 1, 9)))
        draw.ellipse((x0, y0, x1, y1), outline=255, width=ring_width)
    else:
        char = '0123456789ABCDEFGHJKLMNPQRSTUVWXYZ'[int(rng.integers(0, 34))]
        font = font_for_size(max(10, min(80, height)))
        glyph = Image.new('L', (max(16, width * 2), max(16, height * 2)), 0)
        gd = ImageDraw.Draw(glyph)
        box = gd.textbbox((0, 0), char, font=font)
        gd.text((-box[0], -box[1]), char, fill=255, font=font)
        box = glyph.getbbox()
        if box is not None:
            glyph = glyph.crop(box).resize((max(3, width), max(3, height)), RESAMPLE)
            mask.paste(glyph, (x0, y0), glyph)

def make_mask(seed: int, target_coverage: float) -> tuple[Image.Image, float]:
    best_mask, best_coverage, best_error = None, 0.0, float('inf')
    for attempt in range(8):
        rng = np.random.default_rng(seed + 1009 * attempt)
        mask = Image.new('L', (IMAGE_SIZE, IMAGE_SIZE), 0)
        for _ in range(18):
            coverage = float((np.asarray(mask) > 16).mean())
            error = abs(coverage - target_coverage)
            if error < best_error:
                best_mask, best_coverage, best_error = mask.copy(), coverage, error
            if coverage >= target_coverage * 0.96:
                break
            draw_primitive(mask, rng, max(target_coverage - coverage, 0.002))
        coverage = float((np.asarray(mask) > 16).mean())
        error = abs(coverage - target_coverage)
        if error < best_error:
            best_mask, best_coverage, best_error = mask.copy(), coverage, error
        if target_coverage * 0.75 <= coverage <= target_coverage * 1.25:
            return mask, coverage
    return best_mask, best_coverage

def render_synthetic(item: dict, return_coverage: bool = False):
    seed = int(item['seed'])
    rng = np.random.default_rng(seed)
    target = float(item['target_coverage'])
    mask, coverage = make_mask(seed, target)
    mode = item['background_mode']
    if mode == 'dark':
        background = int(rng.integers(0, 19)); foreground = int(rng.integers(max(80, background + 65), 256))
    elif mode == 'light':
        background = int(rng.integers(175, 246)); foreground = int(rng.integers(0, max(1, background - 65)))
    else:
        background = int(rng.integers(25, 101))
        if rng.random() < 0.7:
            foreground = int(rng.integers(min(255, background + 65), 256))
        else:
            foreground = int(rng.integers(0, max(1, background - 20)))
    mask_values = np.asarray(mask, dtype=np.float32) / 255.0
    image = background + mask_values * (foreground - background)
    image = np.clip(np.rint(image), 0, 255).astype(np.uint8)
    tensor = normalize_image(image)
    return (tensor, coverage) if return_coverage else tensor

def load_natural(item: dict, root: Path = BSDS_ROOT) -> torch.Tensor:
    path = root / item['path']
    with Image.open(path) as source:
        image = source.convert('L')
        if item['transform'] == 'full':
            fill = int(np.asarray(image, dtype=np.uint8).mean())
            image = ImageOps.pad(image, (IMAGE_SIZE, IMAGE_SIZE), method=RESAMPLE, color=fill, centering=(0.5, 0.5))
        else:
            crop = item['crop']; size = int(crop['size']); x, y = int(crop['x']), int(crop['y'])
            image = image.crop((x, y, x + size, y + size)).resize((IMAGE_SIZE, IMAGE_SIZE), RESAMPLE)
        if item.get('hflip', False):
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        angle = int(item.get('rotation', 0))
        if angle:
            fill = int(np.asarray(image, dtype=np.uint8).mean())
            image = image.rotate(angle, resample=Image.Resampling.BILINEAR, fillcolor=fill)
        return normalize_image(np.asarray(image, dtype=np.uint8))

def load_item(item: dict) -> torch.Tensor:
    return render_synthetic(item) if item['source'] == 'synthetic' else load_natural(item)

class QuadDataset(Dataset):
    def __init__(self, manifest: Path):
        self.records = [json.loads(line) for line in Path(manifest).read_text(encoding='utf-8').splitlines() if line]
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        quad = torch.stack([load_item(item) for item in self.records[index]['items']]).to(torch.float32)
        if quad.shape != (4, IMAGE_SIZE, IMAGE_SIZE):
            raise RuntimeError(f'Invalid quad shape: {tuple(quad.shape)}')
        return quad

## 2. Новые манифесты

Натуральная часть использует только соответствующий BSDS split. Это сохраняет отсутствие утечки между
train, validation и internal test. В манифесте синтетики сохраняются целевая заполненность и режим фона.

In [ ]:
def image_catalog(split: str):
    rows = []
    for path in sorted((BSDS_ROOT / 'images' / split).glob('*.jpg')):
        with Image.open(path) as image:
            width, height = image.size
        rows.append((path.relative_to(BSDS_ROOT).as_posix(), width, height))
    if not rows:
        raise RuntimeError(f'No BSDS images for split {split}')
    return rows

def synthetic_spec(rng: np.random.Generator) -> dict:
    bin_name = str(rng.choice(OCCUPANCY_NAMES, p=OCCUPANCY_WEIGHTS))
    low, high = OCCUPANCY_RANGES[bin_name]
    target = float(rng.uniform(low, high))
    mode = str(rng.choice(BACKGROUND_NAMES, p=BACKGROUND_WEIGHTS))
    best = None
    for _ in range(12):
        seed = int(rng.integers(1, 2**63 - 1))
        item = {'source': 'synthetic', 'version': 2, 'seed': seed, 'occupancy_bin': bin_name,
                'target_coverage': round(target, 6), 'background_mode': mode}
        _, actual = render_synthetic(item, return_coverage=True)
        item['actual_coverage'] = round(float(actual), 6)
        best = item
        if low <= actual <= high:
            return item
    return best

def natural_spec(rng: np.random.Generator, catalog: list) -> dict:
    path, width, height = catalog[int(rng.integers(0, len(catalog)))]
    full = rng.random() < 0.25
    item = {'source': 'bsds', 'path': path, 'transform': 'full' if full else 'crop',
            'hflip': bool(rng.random() < 0.5), 'rotation': int(rng.choice([-10, -5, 0, 0, 0, 5, 10]))}
    if not full:
        maximum = min(width, height)
        size = int(round(np.exp(rng.uniform(np.log(IMAGE_SIZE), np.log(maximum)))))
        size = int(np.clip(size, IMAGE_SIZE, maximum))
        item['crop'] = {'x': int(rng.integers(0, width - size + 1)),
                        'y': int(rng.integers(0, height - size + 1)), 'size': size}
    return item

def sample_item(rng: np.random.Generator, catalog: list) -> dict:
    return natural_spec(rng, catalog) if rng.random() < NATURAL_FRACTION else synthetic_spec(rng)

def make_manifest(name: str, count: int, bsds_split: str, seed: int) -> Path:
    rng = np.random.default_rng(seed)
    catalog = image_catalog(bsds_split)
    path = OUTPUT_ROOT / 'manifests' / f'{name}.jsonl'
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as stream:
        for index in range(count):
            items, signatures = [], set()
            while len(items) < 4:
                item = sample_item(rng, catalog)
                signature = json.dumps(item, sort_keys=True)
                if signature not in signatures:
                    signatures.add(signature); items.append(item)
            stream.write(json.dumps({'id': f'{name}-{index:05d}', 'items': items}) + '\n')
            if (index + 1) % 2_000 == 0:
                print(f'{name}: {index + 1}/{count}', flush=True)
    return path

manifests = {name: make_manifest(name, count, BSDS_SPLITS[name], SPLIT_SEEDS[name])
             for name, count in SPLIT_COUNTS.items()}
print({name: sum(1 for _ in path.open()) for name, path in manifests.items()})

## 3. Быстрый аудит v2

Аудит проверяет фактические изображения, а не только поля манифеста. Перед обучением должны выполняться:

- натуральная доля близка к 50%;
- доля синтетики с заполнением меньше 5% не превышает 30%;
- очень разреженные изображения сохранены, но больше не доминируют;
- средняя яркость синтетики приблизилась к натуральной;
- нет пустых изображений, и максимум каждого изображения равен 1.

In [ ]:
def image_features(image: torch.Tensor) -> dict:
    array = image.numpy()
    spectrum = np.abs(np.fft.fftshift(np.fft.fft2(array))) ** 2
    yy, xx = np.ogrid[-64:64, -64:64]
    high = np.sqrt(xx**2 + yy**2) >= 32
    return {
        'mean': float(array.mean()),
        'std': float(array.std()),
        'frac_dark': float((array < 0.05).mean()),
        'high_frequency_energy': float(spectrum[high].sum() / (spectrum.sum() + 1e-12)),
    }

def audit_split(name: str, path: Path, quad_limit: int) -> pd.DataFrame:
    records = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]
    indices = np.linspace(0, len(records) - 1, min(quad_limit, len(records)), dtype=int)
    rows = []
    for index in indices:
        record = records[int(index)]
        for plane, item in enumerate(record['items'], start=1):
            image = load_item(item)
            if not torch.isfinite(image).all() or not torch.isclose(image.max(), torch.tensor(1.0), atol=1e-5):
                raise RuntimeError(f'Invalid image in {record["id"]}, plane {plane}')
            row = image_features(image)
            row.update({'split': name, 'sample_id': record['id'], 'plane': plane, 'source': item['source'],
                        'occupancy': float(item.get('actual_coverage', np.nan)),
                        'occupancy_bin': item.get('occupancy_bin', ''),
                        'background_mode': item.get('background_mode', ''),
                        'natural_transform': item.get('transform', '')})
            rows.append(row)
    return pd.DataFrame(rows)

audit = pd.concat([
    audit_split('train', manifests['train'], 2_048),
    audit_split('val', manifests['val'], 512),
    audit_split('test_internal', manifests['test_internal'], 512),
], ignore_index=True)
audit.to_csv(AUDIT_DIR / 'image_audit.csv', index=False)
summary = audit.groupby(['split', 'source'])[['mean', 'std', 'frac_dark', 'high_frequency_energy']].agg(['mean', 'std']).round(4)
display(summary)

train = audit[audit['split'] == 'train']
synthetic = train[train['source'] == 'synthetic']
natural = train[train['source'] == 'bsds']
natural_share = float((train['source'] == 'bsds').mean())
below_3 = float((synthetic['occupancy'] < 0.03).mean())
below_5 = float((synthetic['occupancy'] < 0.05).mean())
synthetic_mean = float(synthetic['mean'].mean())
natural_mean = float(natural['mean'].mean())
synthetic_dark = float(synthetic['frac_dark'].mean())
checks = {
    'natural_share': natural_share,
    'synthetic_below_3pct': below_3,
    'synthetic_below_5pct': below_5,
    'synthetic_mean': synthetic_mean,
    'natural_mean': natural_mean,
    'brightness_gap': abs(synthetic_mean - natural_mean),
    'synthetic_frac_dark': synthetic_dark,
}
print(json.dumps(checks, indent=2))
assert 0.47 <= natural_share <= 0.53, checks
assert 0.04 <= below_3 <= 0.18, checks
assert below_5 <= 0.30, checks
assert 0.12 <= synthetic_mean <= 0.45, checks
assert abs(synthetic_mean - natural_mean) <= 0.35, checks
assert synthetic_dark <= 0.80, checks

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
sample_records = [json.loads(line) for line in manifests['train'].read_text(encoding='utf-8').splitlines()[:2]]
for row, record in enumerate(sample_records):
    for col, item in enumerate(record['items']):
        axes[row, col].imshow(load_item(item), cmap='gray', vmin=0, vmax=1)
        axes[row, col].set_title(f"{item['source']} {item.get('occupancy_bin', item.get('transform', ''))}")
        axes[row, col].axis('off')
fig.tight_layout(); fig.savefig(AUDIT_DIR / 'samples.png', dpi=160); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for source, frame in train.groupby('source'):
    axes[0].hist(frame['mean'], bins=30, alpha=.5, density=True, label=source)
    axes[1].hist(frame['frac_dark'], bins=30, alpha=.5, density=True, label=source)
axes[2].hist(synthetic['occupancy'], bins=30, color='slateblue')
axes[0].set_title('Средняя яркость'); axes[1].set_title('Доля тёмных пикселей'); axes[2].set_title('Заполненность синтетики')
axes[0].legend(); axes[1].legend(); fig.tight_layout(); fig.savefig(AUDIT_DIR / 'distributions.png', dpi=160); plt.show()

metadata = {
    'schema_version': 2,
    'image_size': IMAGE_SIZE,
    'split_counts': SPLIT_COUNTS,
    'natural_fraction': NATURAL_FRACTION,
    'split_seeds': SPLIT_SEEDS,
    'occupancy_ranges': OCCUPANCY_RANGES,
    'occupancy_weights': OCCUPANCY_WEIGHTS.tolist(),
    'background_weights': dict(zip(BACKGROUND_NAMES, BACKGROUND_WEIGHTS.tolist())),
    'natural_full_scene_fraction': 0.25,
    'audit': checks,
}
(OUTPUT_ROOT / 'dataset_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
(AUDIT_DIR / 'summary.json').write_text(json.dumps({'checks': checks, 'status': 'passed'}, indent=2), encoding='utf-8')
archive = shutil.make_archive(str(ARCHIVE_BASE), 'zip', root_dir=OUTPUT_ROOT)
print(f'AUDIT PASSED. Created {archive}: {Path(archive).stat().st_size / 1024**2:.1f} MiB')

## 4. Короткий baseline против эксперимента B на v2

Этот экран не заменяет полное обучение. Он проверяет, сохраняется ли преимущество shared-depth после
изменения данных. Физика, loss, subset и порядок батчей совпадают для обеих моделей.

In [ ]:
class DoubleConv(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU(),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU())

class UpBlock(nn.Module):
    def __init__(self, cin, skip, cout):
        super().__init__(); self.reduce = nn.Conv2d(cin, cout, 1); self.block = DoubleConv(cout + skip, cout)
    def forward(self, x, skip):
        x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        return self.block(torch.cat((self.reduce(x), skip), dim=1))

class BaselineUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e0, self.e1, self.e2, self.e3 = DoubleConv(4, 32), DoubleConv(32, 64), DoubleConv(64, 128), DoubleConv(128, 256)
        self.pool, self.b = nn.MaxPool2d(2), DoubleConv(256, 512)
        self.d3, self.d2, self.d1, self.d0 = UpBlock(512, 256, 256), UpBlock(256, 128, 128), UpBlock(128, 64, 64), UpBlock(64, 32, 32)
        self.head = nn.Conv2d(32, 1, 1)
    def forward(self, x):
        e0 = self.e0(x); e1 = self.e1(self.pool(e0)); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        x = self.b(self.pool(e3)); x = self.d3(x, e3); x = self.d2(x, e2); x = self.d1(x, e1); x = self.d0(x, e0)
        return self.head(x)

class DepthEmbedding(nn.Module):
    def __init__(self, channels):
        super().__init__(); hidden = max(16, channels // 2)
        self.net = nn.Sequential(nn.Linear(1, hidden), nn.SiLU(), nn.Linear(hidden, channels))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)
    def forward(self, depths):
        return self.net(depths[:, None])[:, :, None, None]

class PlaneFusion(nn.Sequential):
    def __init__(self, channels):
        super().__init__(nn.Conv2d(4 * channels, channels, 1, bias=False), nn.GroupNorm(8, channels), nn.SiLU())

class SharedDepthUNet(nn.Module):
    def __init__(self):
        super().__init__(); self.planes = 4; channels = (32, 64, 128, 256, 512)
        self.encoders = nn.ModuleList([DoubleConv(1, 32), DoubleConv(32, 64), DoubleConv(64, 128), DoubleConv(128, 256), DoubleConv(256, 512)])
        self.depth_embeddings = nn.ModuleList([DepthEmbedding(c) for c in channels])
        self.fusions = nn.ModuleList([PlaneFusion(c) for c in channels])
        self.pool = nn.MaxPool2d(2)
        self.d3, self.d2, self.d1, self.d0 = UpBlock(512, 256, 256), UpBlock(256, 128, 128), UpBlock(128, 64, 64), UpBlock(64, 32, 32)
        self.head = nn.Conv2d(32, 1, 1)
        z = torch.tensor(ZS, dtype=torch.float32) * 1e3
        self.register_buffer('depths', (z - z.mean()) / (z.max() - z.min()))
    def forward(self, x):
        batch, planes, height, width = x.shape
        features = x.reshape(batch * planes, 1, height, width); depths = self.depths.repeat(batch); fused = []
        for level, (encoder, embedding, fusion) in enumerate(zip(self.encoders, self.depth_embeddings, self.fusions)):
            features = encoder(features) + embedding(depths).to(features.dtype)
            _, channels, h, w = features.shape
            fused.append(fusion(features.reshape(batch, planes * channels, h, w)))
            if level + 1 < len(self.encoders): features = self.pool(features)
        e0, e1, e2, e3, x = fused
        x = self.d3(x, e3); x = self.d2(x, e2); x = self.d1(x, e1); x = self.d0(x, e0)
        return self.head(x)

class FresnelLayer(nn.Module):
    def __init__(self, device):
        super().__init__(); c = (torch.arange(IMAGE_SIZE, dtype=torch.float64) - IMAGE_SIZE / 2) * PIXEL_PITCH
        u, v = torch.meshgrid(c, c, indexing='ij')
        kernels = [torch.exp(1j * math.pi * (u.square() + v.square()) / (WAVELENGTH * z)).to(torch.complex64) for z in ZS]
        self.register_buffer('kernels', torch.stack(kernels).to(device), persistent=False)
    def forward(self, phase):
        field = torch.polar(torch.ones_like(phase, dtype=torch.float32), phase.float()); outputs = []
        for kernel in self.kernels:
            value = torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(field * kernel, dim=(-2, -1))), dim=(-2, -1)).abs()
            outputs.append(value)
        amplitude = torch.cat(outputs, dim=1)
        return amplitude / (amplitude.amax(dim=(-2, -1), keepdim=True) + 1e-8)

def pearson_cc(a, b, eps=1e-8):
    a = a - a.mean(dim=(-2, -1), keepdim=True); b = b - b.mean(dim=(-2, -1), keepdim=True)
    return (a * b).sum(dim=(-2, -1)) / (torch.sqrt(a.square().sum(dim=(-2, -1)) * b.square().sum(dim=(-2, -1))) + eps)

def doe_loss(target, reconstruction):
    cc = pearson_cc(target, reconstruction)
    sv = torch.stack([ssim(target[:, m:m+1], reconstruction[:, m:m+1], data_range=1.0, size_average=False) for m in range(4)], dim=1)
    return (1 - cc).mean() + 1.1 * (1 - sv).mean(), cc, sv

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Enable Kaggle GPU before the screen: Settings -> Accelerator -> GPU')
device = torch.device('cuda'); physics = FresnelLayer(device); pin_memory = True

full_train = QuadDataset(manifests['train']); full_val = QuadDataset(manifests['val'])
subset_rng = np.random.default_rng(DATA_SEED + 101)
train_indices = subset_rng.choice(len(full_train), size=SCREEN_TRAIN, replace=False).tolist()
val_indices = subset_rng.choice(len(full_val), size=SCREEN_VAL, replace=False).tolist()
train_set, val_set = Subset(full_train, train_indices), Subset(full_val, val_indices)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False; torch.backends.cudnn.benchmark = True

def make_loaders():
    generator = torch.Generator().manual_seed(TRAIN_SEED + 202)
    return (
        DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, generator=generator, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0),
        DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0),
    )

def lr_multiplier(epoch):
    if epoch < 1: return epoch + 1
    progress = (epoch - 1) / max(1, SCREEN_EPOCHS - 1)
    return (1e-5 / LR) + (1 - 1e-5 / LR) * 0.5 * (1 + math.cos(math.pi * progress))

@torch.no_grad()
def validate(model, loader):
    model.eval(); loss_total = 0.0; cc_total = torch.zeros(4, device=device); ss_total = torch.zeros(4, device=device); count = 0
    for target in loader:
        target = target.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda'):
            phase = math.pi * torch.tanh(model(2 * target - 1))
        loss, cc, sv = doe_loss(target, physics(phase.float()))
        loss_total += loss.item() * target.shape[0]; cc_total += cc.sum(0); ss_total += sv.sum(0); count += target.shape[0]
    return loss_total / count, (cc_total / count).cpu().tolist(), (ss_total / count).cpu().tolist()

def run(name):
    print(f'\n===== {name.upper()} / DATASET V2 =====', flush=True)
    set_seed(TRAIN_SEED); train_loader, val_loader = make_loaders()
    model = (BaselineUNet() if name == 'baseline' else SharedDepthUNet()).to(device)
    params = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_multiplier)
    scaler = torch.amp.GradScaler('cuda'); history = []
    for epoch in range(SCREEN_EPOCHS):
        started = time.perf_counter(); model.train(); total = 0.0; seen = 0
        for batch_index, target in enumerate(train_loader, start=1):
            target = target.to(device, non_blocking=True); optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda'):
                phase = math.pi * torch.tanh(model(2 * target - 1))
            with torch.autocast(device_type='cuda', enabled=False):
                loss, _, _ = doe_loss(target, physics(phase.float()))
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); total += loss.item() * target.shape[0]; seen += target.shape[0]
            if batch_index == 1 or batch_index % LOG_EVERY == 0 or batch_index == len(train_loader):
                elapsed = time.perf_counter() - started; eta = elapsed / batch_index * (len(train_loader) - batch_index)
                print(f'{name} epoch {epoch+1}/{SCREEN_EPOCHS}: {batch_index}/{len(train_loader)}, loss={loss.item():.4f}, ETA={eta:.0f}s', flush=True)
        val_loss, cc_planes, ss_planes = validate(model, val_loader); scheduler.step()
        row = {'epoch': epoch + 1, 'train_loss': total / seen, 'val_loss': val_loss,
               'val_cc': float(np.mean(cc_planes)), 'val_ssim': float(np.mean(ss_planes)),
               'val_cc_min': float(np.min(cc_planes)), 'val_ssim_min': float(np.min(ss_planes)),
               'epoch_seconds': time.perf_counter() - started}
        row.update({f'cc_z{m+1}': v for m, v in enumerate(cc_planes)}); row.update({f'ssim_z{m+1}': v for m, v in enumerate(ss_planes)})
        history.append(row); print(row, flush=True)
    with (SCREEN_DIR / f'metrics_{name}.csv').open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=history[0].keys()); writer.writeheader(); writer.writerows(history)
    del model, optimizer, scheduler, scaler, train_loader, val_loader; gc.collect(); torch.cuda.empty_cache()
    return {'history': history, 'parameter_count': params}

results = {name: run(name) for name in ('baseline', 'shared_depth')}
def tail(name, key): return float(np.mean([row[key] for row in results[name]['history'][-3:]]))
summary = {}
for name in results:
    summary[name] = {key: tail(name, key) for key in ('val_cc', 'val_ssim', 'val_cc_min', 'val_ssim_min', 'epoch_seconds')}
    summary[name]['parameter_count'] = results[name]['parameter_count']
delta = {key: summary['shared_depth'][key] - summary['baseline'][key] for key in ('val_cc', 'val_ssim', 'val_cc_min', 'val_ssim_min')}
meaningful = delta['val_cc'] >= .005 or delta['val_ssim'] >= .005
safe = delta['val_cc'] >= -.003 and delta['val_ssim'] >= -.003 and delta['val_cc_min'] >= -.005 and delta['val_ssim_min'] >= -.005
decision = 'PROMISING' if meaningful and safe else ('NOT_PROMISING' if min(delta.values()) < -.01 else 'BORDERLINE')
output = {'dataset_audit': checks, 'baseline': summary['baseline'], 'shared_depth': summary['shared_depth'],
          'delta_shared_minus_baseline': delta, 'decision': decision,
          'config': {'data_seed': DATA_SEED, 'train_seed': TRAIN_SEED, 'epochs': SCREEN_EPOCHS}}
(SCREEN_DIR / 'summary.json').write_text(json.dumps(output, indent=2), encoding='utf-8')
print(json.dumps(output, indent=2))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for name, color in [('baseline', 'tab:blue'), ('shared_depth', 'tab:orange')]:
    epochs = [r['epoch'] for r in results[name]['history']]
    axes[0].plot(epochs, [r['val_loss'] for r in results[name]['history']], marker='o', label=name, color=color)
    axes[1].plot(epochs, [r['val_cc'] for r in results[name]['history']], marker='o', label=name, color=color)
    axes[2].plot(epochs, [r['val_ssim'] for r in results[name]['history']], marker='o', label=name, color=color)
for axis, title in zip(axes, ['Validation loss', 'Validation CC', 'Validation SSIM']):
    axis.set_title(title); axis.grid(alpha=.3); axis.legend(); axis.set_xlabel('epoch')
fig.tight_layout(); fig.savefig(SCREEN_DIR / 'comparison.png', dpi=160); plt.show()
print('Dataset:', str(ARCHIVE_BASE) + '.zip')
print('Audit:', AUDIT_DIR)
print('Screen:', SCREEN_DIR)